In [ ]:
import wandb
wandb.login()

In [ ]:
import os, textwrap
from google.colab import drive, userdata
drive.mount('/content/drive')

os.environ['GH_USER'] = userdata.get('GH_USER')
os.environ['GH_TOKEN'] = userdata.get('GH_TOKEN')

askpass_path = '/tmp/gh_askpass.sh'
with open(askpass_path, 'w') as f:
    f.write(textwrap.dedent('''\
        #!/bin/sh
        case "$1" in
          *Username*) echo "$GH_USER" ;;
          *Password*) echo "$GH_TOKEN" ;;
          *) echo "" ;;
        esac
    '''))
os.chmod(askpass_path, 0o700)
os.environ['GIT_ASKPASS'] = askpass_path
os.environ['GIT_TERMINAL_PROMPT'] = '0'

!git clone https://github.com/incredet/HyperSpectralSuperResolution.git 2>/dev/null || echo 'Repo already cloned'

In [ ]:
!pip install -q -r /content/HyperSpectralSuperResolution/requirements.txt

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU — change runtime type'
print(torch.cuda.get_device_name(0))

In [ ]:
#   CNMF GT:        exp1_rrdb_96x24, exp1_rrdb_192x24,
#                   exp1_essa_dim180, exp1_essa_dim252,
#                   exp1_cst_dim96, exp1_cst_dim180
#   SFIM GT:        exp2_rrdb_192x24_sfim
#   Synthetic:      exp3_rrdb_192x24_synthetic_bicubic,
#                   exp3_rrdb_192x24_synthetic_cnmf,
#                   exp3_cst_dim90_synthetic
CONFIG = 'exp1_rrdb_192x24'

import json, yaml
from pathlib import Path

REPO  = Path('/content/HyperSpectralSuperResolution')
DRIVE = Path('/content/drive/Shareddrives/HyperResData/EMIT_S-2_Matches/2026-04-02')

cfg = yaml.safe_load((REPO / 'training' / 'configs' / f'{CONFIG}.yaml').read_text())
print(f'exp_name:      {cfg["exp_name"]}')
print(f'gt_source:     {cfg["gt_source"]}')
print(f'zip_dir:       {cfg["zip_dir"]}')
print(f'zip_dir_full:  {cfg.get("zip_dir_full", cfg["zip_dir"])}')
print(f'split_json:    {cfg["split_json"]}')

In [ ]:
split = json.loads(Path(cfg['split_json']).read_text())
train_aois = set(split['train'])
val_aois   = set(split['val'])
test_aois  = set(split['test'])
assert not (train_aois & val_aois) and not (train_aois & test_aois) and not (val_aois & test_aois)
print(f'AOI split: {len(train_aois)} train / {len(val_aois)} val / {len(test_aois)} test')

In [ ]:
# Local /content/data/zips_X is mirrored from Drive {DRIVE}/zips_X.
import shutil

local_train = Path(cfg['zip_dir'])
local_full  = Path(cfg.get('zip_dir_full', cfg['zip_dir']))
drive_train = DRIVE / local_train.name
drive_full  = DRIVE / local_full.name

local_train.mkdir(parents=True, exist_ok=True)
for zp in drive_train.glob('*.zip'):
    aoi = zp.stem.split('__')[0]
    if aoi in train_aois and not (local_train / zp.name).exists():
        shutil.copy2(zp, local_train / zp.name)

if local_full != local_train:
    local_full.mkdir(parents=True, exist_ok=True)
    for zp in drive_full.glob('*.zip'):
        aoi = zp.stem.split('__')[0]
        if aoi in (val_aois | test_aois) and not (local_full / zp.name).exists():
            shutil.copy2(zp, local_full / zp.name)

n_train = sum(1 for _ in local_train.glob('*.zip'))
size_train = sum(f.stat().st_size for f in local_train.glob('*.zip')) / 1e9
print(f'Train: {n_train} zips, {size_train:.1f} GB')
if local_full != local_train:
    n_full = sum(1 for _ in local_full.glob('*.zip'))
    size_full = sum(f.stat().st_size for f in local_full.glob('*.zip')) / 1e9
    print(f'Val/test: {n_full} zips, {size_full:.1f} GB')

In [ ]:
RESUME = ''
resume_arg = f'--resume {RESUME}' if RESUME else ''

!cd /content/HyperSpectralSuperResolution/training && \
 python train.py --config configs/{CONFIG}.yaml {resume_arg}

In [ ]:
# Evaluate. CKPT defaults to {out_dir}/{exp_name}-{gt_source}/models/best.pth — adjust if needed.
exp_dir = Path(cfg['out_dir']) / f"{cfg['exp_name']}-{cfg['gt_source']}"
CKPT = str(exp_dir / 'models' / 'best.pth')

!cd /content/HyperSpectralSuperResolution/training && \
 python evaluate.py --config configs/{CONFIG}.yaml --checkpoint {CKPT} --n-vis 8

In [ ]:
from IPython.display import display, Image
fig_dir = exp_dir / 'eval_test' / 'figures'
for png in sorted(fig_dir.glob('*_main.png'))[:8]:
    print(png.name)
    display(Image(filename=str(png), width=800))